# 토픽 골드셋 생성\n\n개념 하나를 anchor로 삼아 토픽을 구성한다.\n\n- **개념 1개 = 토픽 1개**\n- 각 예시/실습은 **같은 `from_topic` 개념에 우선 배정**(거리 무관, 앞 개념 우선)\n- 토픽 매칭 실패 시에만 \"바로 앞 개념 + `MAX_GAP`\" 규칙 적용\n- 개념보다 앞에 나오는 아이템은 첫 번째 개념에 귀속\n\n대상 날짜: `2026-02-09` ~ `2026-02-13` 5개 일괄 처리\n\n→ 날짜별 `*_sequence_goldset.json`</cell id="tg-md-intro">

In [41]:
import json
import re
from pathlib import Path

import pandas as pd
import google.generativeai as genai

import sys
sys.path.insert(0, str(Path(".").resolve().parent))
from app.core.config import settings

genai.configure(api_key=settings.api_key)
MODEL = settings.eval_model

DATES    = [f"2026-02-{d:02d}" for d in range(9, 14)]
# DATES    = [f"2026-02-10"]

GS       = Path("../data/goldset")
GS_LABEL = Path("../data/goldset/labeling")
CSV_PATH = Path("../data/processed/lectures_kss.csv")

df_all = pd.read_csv(CSV_PATH)
print(f"대상 날짜: {', '.join(DATES)}  |  model: {MODEL}\n")
for date in DATES:
    n = int((df_all["date"] == date).sum())
    print(f"  {date}: 문장 {n}개")

대상 날짜: 2026-02-09, 2026-02-10, 2026-02-11, 2026-02-12, 2026-02-13  |  model: models/gemini-2.5-pro

  2026-02-09: 문장 1646개
  2026-02-10: 문장 1815개
  2026-02-11: 문장 1660개
  2026-02-12: 문장 1037개
  2026-02-13: 문장 1543개


In [42]:
# ── 날짜별 세 골드셋 로드 ────────────────────────────────────────────
def load_goldsets(date):
    co_f = json.loads((GS_LABEL / f"{date}_concept_goldset_final.json").read_text(encoding="utf-8"))
    ex_f = json.loads((GS_LABEL / f"{date}_example_goldset_final.json").read_text(encoding="utf-8"))
    pr_f = json.loads((GS_LABEL / f"{date}_practice_goldset_final.json").read_text(encoding="utf-8"))
    return co_f["concepts"], ex_f.get("examples", []), pr_f["practices"]

In [43]:
# ── LLM 기반 토픽 배정 ───────────────────────────────────────────────
def _parse_json(raw):
    text = re.sub(r"^```\w*\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group()) if m else {}


GROUP_SYS = "당신은 강의 콘텐츠를 개념 단위로 분류하는 전문가입니다. 응답은 JSON만 출력하세요."

GROUP_USER = """아래 개념 목록과 예시/실습 목록이 있습니다.
각 예시/실습을 가장 적합한 개념에 배정하세요.

규칙:
- 위치(순서)가 아닌 토픽(내용) 기준으로 판단하세요.
- 정확히 일치하지 않아도 내용상 가장 관련 깊은 개념에 배정하세요.
- 개념 목록 중 어느 것과도 전혀 관련이 없을 때만 concept_index를 null로 반환하세요.

개념 목록 ([인덱스] 개념명):
{concepts}

예시 목록 ([인덱스] 토픽):
{examples}

실습 목록 ([인덱스] 토픽):
{practices}

아래 JSON 형식으로만 응답하세요:
{{"examples": [{{"index": 0, "concept_index": 0}}], "practices": [{{"index": 0, "concept_index": null}}]}}"""


def _label(item, name_key):
    return item.get("from_topic") or item.get(name_key, "")


def llm_assign(concepts_s, examples_s, practices_s):
    if not examples_s and not practices_s:
        return {}, {}
    model = genai.GenerativeModel(MODEL)
    concepts_str  = "\n".join(f"[{i}] {c['concept_name']}" for i, c in enumerate(concepts_s))
    examples_str  = "\n".join(f"[{i}] {_label(e, 'example_name')}"  for i, e in enumerate(examples_s))
    practices_str = "\n".join(f"[{i}] {_label(p, 'practice_name')}" for i, p in enumerate(practices_s))
    prompt = f"{GROUP_SYS}\n\n{GROUP_USER.format(concepts=concepts_str, examples=examples_str, practices=practices_str)}"
    resp = model.generate_content(prompt, generation_config=genai.GenerationConfig(
        response_mime_type="application/json", temperature=0))
    result = _parse_json(resp.text)

    def _parse_map(key):
        out = {}
        for a in result.get(key, []):
            try:
                idx = int(a["index"])
                ci  = a.get("concept_index")
                out[idx] = None if ci is None else int(ci)
            except (KeyError, ValueError, TypeError):
                continue
        return out

    return _parse_map("examples"), _parse_map("practices")


def build_buckets(concepts, examples, practices):
    concepts_s  = sorted(concepts,  key=lambda x: x["start"])
    examples_s  = sorted(examples,  key=lambda x: x["start"])
    practices_s = sorted(practices, key=lambda x: x["start"])

    ex_map, pr_map = llm_assign(concepts_s, examples_s, practices_s)

    buckets = [{"concept": c, "examples": [], "practices": []} for c in concepts_s]
    dropped_ex, dropped_pr = [], []

    for i, e in enumerate(examples_s):
        ci = ex_map.get(i)
        if ci is not None and 0 <= ci < len(buckets):
            buckets[ci]["examples"].append(e)
        else:
            dropped_ex.append({"item": e, "reason": "해당 개념 없음"})

    for i, p in enumerate(practices_s):
        ci = pr_map.get(i)
        if ci is not None and 0 <= ci < len(buckets):
            buckets[ci]["practices"].append(p)
        else:
            dropped_pr.append({"item": p, "reason": "해당 개념 없음"})

    return buckets, dropped_ex, dropped_pr

In [44]:
# ── 버킷 → 토픽 객체 구성 ───────────────────────────────────────────
def build_topics(buckets):
    topics = []
    for b in buckets:
        c = b["concept"]
        ex_items = sorted(b["examples"],  key=lambda x: x["start"])
        pr_items = sorted(b["practices"], key=lambda x: x["start"])
        all_ends = [c["end"]] + [e["end"] for e in ex_items] + [p["end"] for p in pr_items]

        topics.append({
            "topic_name": c["concept_name"],
            "span_start": c["start"],
            "span_end":   max(all_ends),
            "concepts": [{
                "concept_name": c["concept_name"],
                "start": c["start"], "end": c["end"],
                "key_sentence": c.get("key_sentence", ""),
                "text": c.get("text", ""),
            }],
            "examples": [{
                "example_name": e["example_name"],
                "start": e["start"], "end": e["end"],
                "key_sentence": e.get("key_sentence", ""),
                "text": e.get("text", ""),
            } for e in ex_items],
            "practices": [{
                "practice_name": p["practice_name"],
                "start": p["start"], "end": p["end"],
                "key_sentence": p.get("key_sentence", ""),
                "text": p.get("text", ""),
            } for p in pr_items],
        })
    return topics

In [45]:
# ── 5개 날짜 일괄 처리 + JSON 저장 ───────────────────────────────────
all_topics = {}
for date in DATES:
    concepts, examples, practices = load_goldsets(date)
    buckets, dropped_ex, dropped_pr = build_buckets(concepts, examples, practices)
    topics = build_topics(buckets)
    all_topics[date] = topics

    full = [t for t in topics if t["examples"] and t["practices"]]
    n_assigned_ex = sum(len(b["examples"]) for b in buckets)
    n_assigned_pr = sum(len(b["practices"]) for b in buckets)

    sequence_goldset = {
        "date": date,
        "method": "llm_topic_based",
        "topic_count": len(topics),
        "concept_count": len(concepts),
        "example_count": len(examples),
        "practice_count": len(practices),
        "topics": topics,
    }
    json_path = GS / f"{date}_sequence_goldset.json"
    json_path.write_text(json.dumps(sequence_goldset, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[{date}]  개념 {len(concepts)} / 예시 {len(examples)} / 실습 {len(practices)}")
    print(f"  배정: 예시 {n_assigned_ex} / 실습 {n_assigned_pr}  |  제외: 예시 {len(dropped_ex)} / 실습 {len(dropped_pr)}")
    print(f"  토픽 {len(topics)}개 (개념+예시+실습 모두: {len(full)}개)  →  {json_path.name}")
    if dropped_ex or dropped_pr:
        for d in dropped_ex:
            e = d["item"]
            print(f"    제외 예시 [{e['start']}~{e['end']}] {e['example_name'][:40]}")
        for d in dropped_pr:
            p = d["item"]
            print(f"    제외 실습 [{p['start']}~{p['end']}] {p['practice_name'][:40]}")
    print()

print("완료. 날짜별 *_sequence_goldset.json 저장됨.")

[2026-02-09]  개념 22 / 예시 15 / 실습 22
  배정: 예시 14 / 실습 22  |  제외: 예시 1 / 실습 0
  토픽 22개 (개념+예시+실습 모두: 1개)  →  2026-02-09_sequence_goldset.json
    제외 예시 [1542~1607] 식별/비식별 관계 예시 (Country, City, CountryLang

[2026-02-10]  개념 17 / 예시 15 / 실습 26
  배정: 예시 11 / 실습 16  |  제외: 예시 4 / 실습 10
  토픽 17개 (개념+예시+실습 모두: 4개)  →  2026-02-10_sequence_goldset.json
    제외 예시 [1050~1056] IFNULL 함수 사용법
    제외 예시 [1079~1087] 여러 인자를 받는 NULL 처리 함수 (COALESCE)
    제외 예시 [1103~1103] NULL 값 처리 실생활 예시 (연락처)
    제외 예시 [1193~1195] EXPLAIN의 세 가지 출력 포맷
    제외 실습 [415~478] Q1 테이블 생성 및 데이터 입력
    제외 실습 [978~1016] Q10. STRAIGHT_JOIN 사용
    제외 실습 [1209~1275] EXPLAIN으로 JOIN 쿼리 실행 계획 분석
    제외 실습 [1276~1282] 다른 포맷(JSON, TABLE)으로 실행 계획 출력
    제외 실습 [1283~1319] 인덱스 유무에 따른 쿼리 비용 비교 확인
    제외 실습 [1386~1401] EXPLAIN 포맷 변경하여 실행하기
    제외 실습 [1571~1584] City 테이블 상위 10개 도시 조회 및 전체 개수 확인
    제외 실습 [1587~1619] City 테이블의 NULL 및 빈 값 확인
    제외 실습 [1641~1665] Q2: Country 테이블 카운트 집계로 무결성 확인
    제외 실습 [1666~1687] Q3: 국가 테이블에서 특정 컬럼 조회

[2026-02